# Statistical analysis of distance and TE relationships

**Notebook contents:**
* Extraction of distance matrix from D'orsogna phenotypes
* Conduct linear mixed model on sample
* Conduct distance correlation per phenotype

In [ ]:
import joblib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import numpy as np
import dcor
import statsmodels.formula.api as smf
import os
from main import DorsognaTE
from utils import proper_phenotype_names

# Preliminaries

In [ ]:
phenotype_dict = {
    'singlemill':[0.5,0.1],
    'doublemill':[0.9,0.5],
    'doublering':[0.5,0.5],
    'collswarm':[0.5,0.9],
    'escapesymm':[2,0.9],
    'escapeunsymm':[2,2],
    'escapecoll':[2,3]
    }

dorsogna_TE_exports_dir = 'csvs_actual_te_values'
dorsogna_info_dir = 'simulations_dorsogna_info'

# Extract distance matrix

In [ ]:
def compute_distance_matrix(model, t):
    positions = model.pos[t]
    distance_matrix = cdist(positions, positions, metric='euclidean')
    return distance_matrix

In [ ]:
for phenotype_name, values in phenotype_dict.items():
    C, l = values
    os.makedirs(dorsogna_info_dir, exist_ok=True)
    insights_path = os.path.join(dorsogna_info_dir, phenotype_name)
    model = DorsognaTE(outdir=insights_path)
    model.develop_model(
            C=C,
            l=l,
            phenotype_name=phenotype_name,
            particle_count=100,
            t_max=40,
            vel_scale=0.1,
            fps=20,
            dt=0.1,
            animate=False
        )
    distance_matrices = np.zeros((400, 100, 100))

    for t in range(400):
        distance_matrices[t] = compute_distance_matrix(model, t)
    
    joblib.dump(distance_matrices, f'{insights_path}/dist_{phenotype_name}_{C}_{l}.joblib')

# Linear mixed model

In [ ]:
def to_long(df, value_name):
    long = df.stack().reset_index()
    long.columns = ['timestep', 'pair_id', value_name]
    return long

## Linear velocity

In [ ]:

all_df = pd.DataFrame(columns = ['timestep', 'pair_id', 'distance', 'TE'])

print(f'Loading data...\n')
for key, value in phenotype_dict.items():
    C, l = value
    
    # Get dist df
    jb_file = joblib.load(f'{dorsogna_info_dir}/{key}/dist_{key}_{C}_{l}.joblib')
    t_size, i_size, j_size = jb_file.shape
    reshaped_data = jb_file.reshape(t_size, -1)
    col_names = [f'{i}-{j}' for i in range(i_size) for j in range(j_size)]
    dist_df = pd.DataFrame(reshaped_data, columns=col_names) # create df
    to_drop = [f'{i}-{i}' for i in range(i_size)] # remove diagonals
    dist_df.drop(columns=to_drop, inplace=True)

    # Get TE df
    TE_df = pd.read_csv(f'{dorsogna_TE_exports_dir}/{key}/TElog_{key}_{C}_{l}_linvel_k15.csv', index_col = 0)

    df_dist_long = to_long(dist_df, 'distance')
    df_te_long = to_long(TE_df, 'TE')
    full_df = pd.merge(df_dist_long, df_te_long, on=['timestep', 'pair_id'])
    full_df['phenotype'] = key

    # get a sample only (filters also yung mga nonzero TE)
    full_df = full_df[full_df['TE'] != 0][full_df['distance'] != 0].sample(n=10000) 

    all_df = pd.concat([all_df, full_df])

all_df['TE'] = all_df['TE'].apply(abs)

all_df['distance'] = pd.to_numeric(all_df['distance'], errors='coerce')
all_df = all_df.dropna(subset=['distance'])

model = smf.mixedlm("TE ~ np.log(distance)", all_df, groups=all_df["phenotype"])
result = model.fit()

print(result.summary())

### angular velocity

In [ ]:
all_df = pd.DataFrame(columns = ['timestep', 'pair_id', 'distance', 'TE'])

print(f'Loading data...\n')
for key, value in phenotype_dict.items():
    C, l = value
    # Get dist df
    jb_file = joblib.load(f'{dorsogna_info_dir}/{key}/dist_{key}_{C}_{l}.joblib')
    t_size, i_size, j_size = jb_file.shape
    reshaped_data = jb_file.reshape(t_size, -1)
    col_names = [f'{i}-{j}' for i in range(i_size) for j in range(j_size)] # pre-generate column names
    dist_df = pd.DataFrame(reshaped_data, columns=col_names) # create df
    to_drop = [f'{i}-{i}' for i in range(i_size)] # remove diagonals
    dist_df.drop(columns=to_drop, inplace=True)

    # Get TE df
    TE_df = pd.read_csv(f'{dorsogna_TE_exports_dir}/{key}/TElog_{key}_{C}_{l}_angvel_k15.csv', index_col = 0)

    df_dist_long = to_long(dist_df, 'distance')
    df_te_long = to_long(TE_df, 'TE')
    full_df = pd.merge(df_dist_long, df_te_long, on=['timestep', 'pair_id'])
    full_df['phenotype'] = key

    # get a sample only (filters also yung mga nonzero TE)
    full_df = full_df[full_df['TE'] != 0][full_df['distance'] != 0].sample(n=10000) 

    all_df = pd.concat([all_df, full_df])

all_df['TE'] = all_df['TE'].apply(abs)

all_df['distance'] = pd.to_numeric(all_df['distance'], errors='coerce')
# Then drop any NaNs created by 'coerce'
all_df = all_df.dropna(subset=['distance'])

model = smf.mixedlm("TE ~ np.log(distance)", all_df, groups=all_df["phenotype"])
result = model.fit()

print(result.summary())

# Distance correlation

## Linear velocity

In [ ]:
for key, value in phenotype_dict.items():
    print(f"{proper_phenotype_names[key]} ({value}) ===================")
    C, l = value
    # Get dist df
    try:
        jb_file = joblib.load(f'{dorsogna_info_dir}/{key}/dist_{key}_{C}_{l}.joblib')
    except:
        print(f'{dorsogna_info_dir}/{key}/dist_{key}_{C}_{l}.joblib found')
        # continue
    t_size, i_size, j_size = jb_file.shape
    reshaped_data = jb_file.reshape(t_size, -1)
    col_names = [f'{i}-{j}' for i in range(i_size) for j in range(j_size)] # pre-generate column names
    dist_df = pd.DataFrame(reshaped_data, columns=col_names) # create df
    to_drop = [f'{i}-{i}' for i in range(i_size)] # remove diagonals
    dist_df.drop(columns=to_drop, inplace=True)

    # Get TE df
    TE_df = pd.read_csv(f'{dorsogna_TE_exports_dir}/{key}/TElog_{key}_{C}_{l}_linvel_k15.csv', index_col = 0)

    try:
        correlation = dcor.distance_correlation(dist_df.values.flatten(), TE_df.values.flatten()) # independent of timestep

    except Exception as e:
        print(f"An error occurred: {e}")
        continue
    print(f"Distance Correlation: {correlation:.4f}")

    # To check if it's statistically significant:
    test_result = dcor.independence.distance_correlation_t_test(dist_df.values.flatten(), TE_df.values.flatten())
    print(f"p-value: {test_result.p_value:.4f}")

## Angular velocity

In [ ]:
for key, value in phenotype_dict.items():
    print(f"{proper_phenotype_names[key]} ({value}) ===================")
    C, l = value
    # Get dist df
    try:
        jb_file = joblib.load(f'{dorsogna_info_dir}/{key}/dist_{key}_{C}_{l}.joblib')
    except:
        print(f'{dorsogna_info_dir}/{key}/dist_{key}_{C}_{l}.joblib found')
        # continue
    t_size, i_size, j_size = jb_file.shape
    reshaped_data = jb_file.reshape(t_size, -1)
    col_names = [f'{i}-{j}' for i in range(i_size) for j in range(j_size)] # pre-generate column names
    dist_df = pd.DataFrame(reshaped_data, columns=col_names) # create df
    to_drop = [f'{i}-{i}' for i in range(i_size)] # remove diagonals
    dist_df.drop(columns=to_drop, inplace=True)

    # Get TE df
    TE_df = pd.read_csv(f'{dorsogna_TE_exports_dir}/{key}/TElog_{key}_{C}_{l}_angvel_k15.csv', index_col = 0)

    try:
        correlation = dcor.distance_correlation(dist_df.values.flatten(), TE_df.values.flatten()) # independent of timestep

    except Exception as e:
        print(f"An error occurred: {e}")
        continue
    print(f"Distance Correlation: {correlation:.4f}")


    # To check if it's statistically significant:
    test_result = dcor.independence.distance_correlation_t_test(dist_df.values.flatten(), TE_df.values.flatten())
    print(f"p-value: {test_result.p_value:.4f}")